In [1]:
# Load required modules
from pynq import Overlay, allocate, GPIO, MMIO, DefaultIP
import numpy as np
from input_read import *
import time
overlay  = Overlay('design_1.bit')
overlay.download()
reconstruct = overlay.reconstruct_0
list(overlay.ip_dict.keys())

['reconstruct_0/s_axi_control',
 'reconstruct_0/s_axi_scalar_data',
 'zynq_ultra_ps_e_0']

In [2]:
# Step 2: Initialize MMIO interfaces manually
saxi_control = MMIO(0xA0000000, 0xFFFF)
saxi_data = MMIO(0xA0010000, 0xFFFF)
print("MMIO interfaces initialized")

MMIO interfaces initialized


In [3]:
# Step 3: Define register offsets
# Control registers
REG_ap_start = 0 
REG_ap_done = 0 
REG_atomLocations_offset_1 = 0x10 # 16 
REG_atomLocations_offset_2 = 0x14 # 20 
REG_imageProjs_local_offset_1 = 0x1c # 28 
REG_imageProjs_local_offset_2 = 0x20 # 32 
REG_imageProjs_offset_1 = 0x28 # 40
REG_imageProjs_offset_2 = 0x2c # 44
REG_imageProjs_local_size_offset_1 = 0x34 # 52 
REG_imageProjs_local_size_offset_2 = 0x38 # 56 
REG_fullImage_offset_1 = 0x40 # 64 
REG_fullImage_offset_2 = 0x44 # 68 
REG_emissions_offset_1 = 0x4c # 76 
REG_emissions_offset_2 = 0x50 # 80 

# Data registers
REG1_atomLocationsSize = 0x10 # 16 
REG1_projShape0 = 0x18 # 24 
REG1_projShape1 = 0x20 # 32 
REG1_psfSupersample = 0x28 # 40 
REG1_imageProjectionSize = 0x30 # 48 
REG1_fullImage_rows = 0x38 # 56 
REG1_fullImage_cols = 0x40 # 64 
# REG1_emission_cnt = 72 
# REG1_emission_cnt_ctrl = 76

print("Register offsets defined")

Register offsets defined


In [4]:
# Step 4: Allocate memory buffers
atomLocations = allocate(shape=(100000,), dtype=np.float32)
emissions = allocate(shape=(100000,), dtype=np.float32)
fullImage = allocate(shape=(1100000,), dtype=np.float32) 
imageProjs = allocate(shape=(200000,), dtype=np.float32)
imageProjs_local = allocate(shape=(400000,), dtype=np.float32)
imageProjs_local_size = allocate(shape=(100000,), dtype=np.int32)

In [5]:
# Step 5: Define helper functions
def write_offset(addr_low, addr_high, val):
    """Write 64-bit address to two 32-bit registers"""
    value = int(val)
    low_32 = value & 0xFFFFFFFF
    high_32 = (value >> 32) & 0xFFFFFFFF
    saxi_control.write_reg(addr_low, low_32)
    saxi_control.write_reg(addr_high, high_32)
    print(f"Written address {hex(value)} to registers {addr_low}/{addr_high}")

def read_addr(addr_low, addr_high):
    """Read 64-bit address from two 32-bit registers"""
    low = saxi_control.read(addr_low)
    high = saxi_control.read(addr_high)
    return (high << 32 & 0xFFFFFFFF00000000) + low

def copyto(dst, src):
    """Copy source array to destination buffer with float32 conversion"""
    # Convert to float32 if the destination is float32
    if dst.dtype == np.float32 and src.dtype != np.float32:
        src_converted = src.astype(np.float32)
        np.copyto(dst[:len(src_converted)], src_converted)
        print(f"Copied {len(src)} elements to buffer (converted to float32)")
    else:
        np.copyto(dst[:len(src)], src)
        print(f"Copied {len(src)} elements to buffer")

In [6]:
# Step 6: Initialize register addresses
print("\n=== Initializing Register Addresses ===")
write_offset(REG_atomLocations_offset_1, REG_atomLocations_offset_2, atomLocations.physical_address)
write_offset(REG_imageProjs_local_offset_1, REG_imageProjs_local_offset_2, imageProjs_local.physical_address)
write_offset(REG_imageProjs_offset_1, REG_imageProjs_offset_2, imageProjs.physical_address)
write_offset(REG_imageProjs_local_size_offset_1, REG_imageProjs_local_size_offset_2, imageProjs_local_size.physical_address)
write_offset(REG_fullImage_offset_1, REG_fullImage_offset_2, fullImage.physical_address)
write_offset(REG_emissions_offset_1, REG_emissions_offset_2, emissions.physical_address)


=== Initializing Register Addresses ===
Written address 0x79d80000 to registers 16/20
Written address 0x7a800000 to registers 28/32
Written address 0x7a700000 to registers 40/44
Written address 0x7a180000 to registers 52/56
Written address 0x7a200000 to registers 64/68
Written address 0x7a100000 to registers 76/80


In [7]:
# Step 7: Parse input files
print("\n=== Parsing Input Files ===")
input_filename = 'restoutput.txt'  # Change this to your actual file
fullimage_filename = 'fullImage_output.txt'  # Change this to your actual file

# Start timing
start_time = time.time()
input_data, input_status = parse_input_file(input_filename)
full_image, fullimg_status = parse_fullImage(fullimage_filename)
# End timing
end_time = time.time()
print("Input data parsed successfully")
print(f"Parsing time: {end_time - start_time:.6f} seconds")
print(f"Atom locations size: {input_data['atomLocationsSize']}")
print(f"Projection shape: {input_data['projShape0']} x {input_data['projShape1']}")
print(f"PSF supersample: {input_data['psfSupersample']}")
print(f"Image projection size: {input_data['imageProjectionSize']}")
print(f"Full image size: {input_data['fullImage_rows']} x {input_data['fullImage_cols']}")


=== Parsing Input Files ===
Input data parsed successfully
Parsing time: 0.402487 seconds
Atom locations size: 256
Projection shape: 31 x 31
PSF supersample: 1
Image projection size: 1
Full image size: 400 x 400


In [8]:
# Step 8: Copy data to buffers
print("\n=== Copying Data to Buffers ===")

# Start timing
start_time = time.time()
# Copy atomic position data
atom_count = input_data['atomLocationsSize']
for i in range(atom_count):
    loc = input_data['atomLocations'][i]
    atomLocations[2*i] = np.float32(loc.x)      # Explicit float32 conversion
    atomLocations[2*i+1] = np.float32(loc.y)    # Explicit float32 conversion
print(f"Copied {atom_count} atom locations (float32)")

# Copy projection data
copyto(imageProjs, input_data['imageProjs'])
# copyto(imageProjs_local, input_data['imageProjs_local'])
copyto(imageProjs_local_size, input_data['imageProjs_local_size'])

imageProjs_local_packed = pack_floats_to_512bit(input_data['imageProjs_local'])
fullImage_packed = pack_floats_to_512bit(full_image)

# Copy packed data to buffers
copyto(imageProjs_local, imageProjs_local_packed)
copyto(fullImage, fullImage_packed)

# Copy full image data
# copyto(fullImage, full_image)
# End timing
end_time = time.time()
print(f"Copy data time: {end_time - start_time:.6f} seconds")


=== Copying Data to Buffers ===
Copied 256 atom locations (float32)
Copied 100 elements to buffer (converted to float32)
Copied 100 elements to buffer
Copied 100000 elements to buffer
Copied 160000 elements to buffer
Copy data time: 0.014362 seconds


In [9]:
# Step 9: Flush caches to ensure data is written to memory
print("\n=== Flushing Caches ===")
atomLocations.flush()
imageProjs.flush()
imageProjs_local.flush()
imageProjs_local_size.flush()
fullImage.flush()
print("All caches flushed")


=== Flushing Caches ===
All caches flushed


In [10]:
# Step 10: Configure registers
print("\n=== Configuring Registers ===")
# Start timing
start_time = time.time()
saxi_data.write_reg(REG1_atomLocationsSize, int(input_data['atomLocationsSize']))
saxi_data.write_reg(REG1_projShape0, int(input_data['projShape0']))
saxi_data.write_reg(REG1_projShape1, int(input_data['projShape1']))
saxi_data.write_reg(REG1_psfSupersample, int(input_data['psfSupersample']))
saxi_data.write_reg(REG1_imageProjectionSize, int(input_data['imageProjectionSize']))
saxi_data.write_reg(REG1_fullImage_rows, int(input_data['fullImage_rows']))
saxi_data.write_reg(REG1_fullImage_cols, int(input_data['fullImage_cols']))
# End timing
end_time = time.time()
print(f"Configuration registers written: {end_time - start_time:.6f} seconds")


=== Configuring Registers ===
Configuration registers written: 0.001629 seconds


In [11]:
saxi_control.write_reg(REG_ap_start, 0x1)
start_time = time.perf_counter()  # Higher precision timer
while True:
    if(saxi_control.read(REG_ap_done) == 0x4):
        break
elapsed_time = time.perf_counter() - start_time
print(f"IP execution completed in {elapsed_time:.6f} seconds")
print(f"IP execution completed in {elapsed_time*1000:.3f} milliseconds")

IP execution completed in 0.002318 seconds
IP execution completed in 2.318 milliseconds


In [12]:
CTRL = 0x00            # ap_ctrl_hs register offset
AP_START = 1 << 0
AP_DONE  = 1 << 1
AP_IDLE  = 1 << 2
AP_READY = 1 << 3

# Step: start IP
print("\n=== Starting IP Execution ===")
saxi_control.write_reg(CTRL, AP_START)

t0 = time.perf_counter()

# Poll until done (or idle/ready if you prefer)
timeout_s = 2.0  # pick something reasonable for your design
while True:
    ctrl = saxi_control.read(CTRL)
    if ctrl & AP_DONE:          # NOTE: mask bit1 only, not 0x3
        break
    if (time.perf_counter() - t0) > timeout_s:
        raise TimeoutError("IP did not assert ap_done within timeout")

elapsed = time.perf_counter() - t0
print(f"IP execution completed in {elapsed:.6f} s ({elapsed*1000:.3f} ms)")


=== Starting IP Execution ===
IP execution completed in 0.002240 s (2.240 ms)


In [13]:
# Copy results
time.sleep(2) 
result_emissions = np.copy(emissions)
# show all values (no truncation)
np.set_printoptions(threshold=np.inf)
# print(f"Results obtained: {emissions_cnt} emissions")
print(f"All emission values: {result_emissions[:257]}")

All emission values: [31702.482 31734.127 31832.959 33235.5   31759.602 33440.555 33353.75
 33088.793 31695.617 33063.32  31756.797 33676.3   31834.643 33339.227
 31718.129 31684.285 31726.027 33358.543 33143.684 31835.393 33607.11
 33818.27  31838.855 33470.793 33433.285 31835.328 31786.777 31793.037
 33492.176 33312.664 31825.965 31793.537 31660.701 33220.46  31835.502
 31903.578 33632.797 31821.152 31894.621 31773.629 31825.316 33140.07
 33593.965 31707.902 31745.537 31934.969 33194.64  33328.074 33362.54
 33384.727 31761.484 33529.297 33474.402 33567.24  33525.594 31726.082
 31780.81  31777.416 33067.18  31896.785 33416.96  33108.71  31757.848
 31768.078 31698.309 33128.758 31977.34  31810.562 33480.203 31848.39
 31677.848 31801.54  33013.695 31793.22  31610.1   33251.43  31882.506
 31923.531 33337.734 31828.633 31797.406 31760.701 31845.61  31808.111
 31864.17  31865.113 31775.008 31619.316 31770.348 31651.705 31878.832
 33676.023 31792.57  33297.055 33441.2   33251.17  33233.68  

In [14]:
# At the very end of your processing
print("\n=== Final Cleanup ===")

# Stop the IP
saxi_control.write(REG_ap_start, 0x0) # Clear start bit

# Reset all control registers
register_pairs = [
    (REG_atomLocations_offset_1, REG_atomLocations_offset_2),
    (REG_imageProjs_local_offset_1, REG_imageProjs_local_offset_2),
    (REG_imageProjs_offset_1, REG_imageProjs_offset_2),
    (REG_imageProjs_local_size_offset_1, REG_imageProjs_local_size_offset_2),
    (REG_fullImage_offset_1, REG_fullImage_offset_2),
    (REG_emissions_offset_1, REG_emissions_offset_2)
]

for reg1, reg2 in register_pairs:
    saxi_control.write(reg1, 0x0)
    saxi_control.write(reg2, 0x0)

# Reset data registers
data_registers = [
    REG1_atomLocationsSize, REG1_projShape0, REG1_projShape1,
    REG1_psfSupersample, REG1_imageProjectionSize,
    REG1_fullImage_rows, REG1_fullImage_cols
]

for reg in data_registers:
    saxi_data.write(reg, 0x0)

print("All registers reset to clean state")
print("Safe to disconnect from FPGA")


=== Final Cleanup ===
All registers reset to clean state
Safe to disconnect from FPGA
